# X2 — FaithBench 외부 평가 (사람 정답, 최신 LLM 요약) · 보고용

## 왜 하나

M1·K1·K2는 **최신 LLM 응답을 Claude(Opus)가 매긴 정답**으로 평가한다. 생성도 Claude, 정답도 Claude라서 "Claude 기준을 배운 것 아니냐"는 의문이 남는다.
[FaithBench](https://github.com/vectara/FaithBench)(Vectara, NAACL 2025)는 **최신 LLM 10종**(GPT-4o, Claude-3.5-Sonnet, Gemini-1.5-Flash, Llama-3.1 등)이 쓴 요약 750개에 **사람 전문가가 환각 구간**을 표시한 데이터라,
Claude 정답에 기대지 않고 판정기를 잴 수 있다. 공개 판정기들이 거의 맞히지 못하도록 **어려운 사례를 골라 만든** 데이터라 절대 수치는 낮게 나오는 것이 정상이다.

## 무엇을 하나 (학습 없음, API 비용 없음)

- 근거 = FaithBench 원문, 응답 = 요약. 게이트웨이 `judge.py`로 문장마다 8토큰 + 문장 끝 지점을 판정(M1과 같은 코드)
- 기존 조합(F1 base + I2, 임계값 A 2.992). **K2 판정기(`k2/judge_base`)가 있으면 새 조합도**(임계값 A·B는 `k2/k2_summary.json`)
- 비교 대상: FaithBench가 함께 제공하는 공개 판정기 예측(HHEM-2.1 점수, GPT-4o 등 LLM 판정·TrueTeacher·TRUE의 0/1)
- RAGTruth 학습 요약 근거와 8-gram이 50% 이상 겹치는 FaithBench 근거는 주 분석에서 뺀다(학습 오염 차단)

## 사전 정의 (SSOT §6 X2, 실행 전 고정 — 보고용, 판정기·임계값 결정에 쓰지 않음)

- **정답**: 요약마다 최악 등급(FaithBench `binarize.py`의 worst). **환각 = Unwanted가 하나라도**, **멀쩡 = 표시 없음 또는 Benign만**, Questionable이 최악인 요약은 주 분석에서 뺌(보고에서는 환각으로 셈)
- **주 지표**: 응답 최대 조합 점수의 AUROC, 근거 단위 군집 부트스트랩 95% 구간(2,000회, 시드 2034)
- **해석 규칙**: 기존 조합 AUROC − HHEM-2.1 AUROC의 구간이 0보다 크면 "공개 판정기(HHEM-2.1)보다 앞섬", 0을 포함하면 "비슷", 0보다 작으면 "뒤짐". 기존 조합 AUROC 구간 하한이 0.5보다 크지 않으면 "FaithBench에서 판별력 확인 안 됨"
- 보고만: F1·I2 단독 AUROC, 임계값 A(·B)에서 균형 정확도·환각 탐지율·멀쩡한 응답 표시율, 첫 Unwanted 구간 이후 탐지, 생성 모델별 AUROC, Questionable 포함 AUROC, LLM 판정기들의 균형 정확도, 새 조합 − 기존 조합 AUROC 차이

> 런타임: **GPU**(아무거나). 판정 약 10~20분. 끊기면 다시 모두 실행(점수는 드라이브 `x2/`에서 이어서). 끝나면 `x2_results.zip`(원문 없음)이 내려받아진다.


## 1. 환경 준비

In [ ]:
SRC = {'judge': '"""문장 판정기.\n\n입력 방식은 판정기 폴더의 `judge_config.json`이 정한다 (SPEC §4).\n  retrieval (F1 base): 근거 → 40단어 조각 → 임베딩 (요청당 1회, prepare)\n                       판정 텍스트 → 임베딩 → 유사한 조각 top-k (원문 순서) → 교차 인코더 [근거 조각, 문장 앞부분]\n  full      (I2식)    : 교차 인코더 [근거 전체, 응답 처음부터 판정 지점까지]. 근거 검색 없음\n점수는 logit(클수록 환각). 임계값은 재판정 간격(stride)별로 설정 파일의 값을 쓴다.\n"""\nimport json\nimport os\nimport threading\nimport time\nfrom dataclasses import dataclass\n\nimport torch\n\n\n@dataclass\nclass Verdict:\n    score: float          # logit, 클수록 환각\n    hallucinated: bool\n    judge_ms: float       # 검색 + 판정 시간\n\n\n@dataclass\nclass RefIndex:\n    chunks: list\n    emb: object           # torch.Tensor [n_chunks, d] (retrieval) / None (full)\n    prepare_ms: float\n    text: str = \'\'        # 근거 원문 (full 방식)\n\n\ndef chunk_text(text, n):\n    w = text.split()\n    return [\' \'.join(w[i:i + n]) for i in range(0, max(len(w), 1), n)] or [text]\n\n\ndef _stride_key(stride):\n    return \'end\' if not stride else str(stride)\n\n\ndef read_config(model_dir):\n    p = os.path.join(model_dir, \'judge_config.json\')\n    return json.load(open(p)) if os.path.isfile(p) else {}\n\n\ndef input_mode(cfg):\n    """설정 파일에서 입력 방식을 정한다. 명시(`input_mode`)가 없으면 I2 노트북 형식(retrieval: null)을 full로 본다."""\n    if cfg.get(\'input_mode\'):\n        return cfg[\'input_mode\']\n    inp = cfg.get(\'input\') or {}\n    return \'full\' if isinstance(inp, dict) and \'retrieval\' in inp and inp[\'retrieval\'] is None else \'retrieval\'\n\n\nclass StudentJudge:\n    def __init__(self, model_dir, embedder=\'sentence-transformers/all-MiniLM-L6-v2\', k=8, max_len=512,\n                 chunk_words=40, threshold=0.0, device=\'\', mode=\'retrieval\', thresholds=None, threshold_key=\'svc10_orig\',\n                 default_stride=0):\n        from transformers import AutoConfig, AutoModelForSequenceClassification, AutoTokenizer\n        self.device = device or (\'cuda\' if torch.cuda.is_available() else \'cpu\')\n        self.mode = mode\n        self.tok = AutoTokenizer.from_pretrained(model_dir)\n        cfg = AutoConfig.from_pretrained(model_dir)\n        kw = {}\n        if getattr(cfg, \'model_type\', \'\') == \'modernbert\':\n            # 새 transformers는 reference_compile을 모델 인자로 받지 않으므로 설정에 넣는다\n            if hasattr(cfg, \'reference_compile\'):\n                cfg.reference_compile = False\n            kw[\'attn_implementation\'] = \'sdpa\'\n        # 가중치가 fp16으로 저장돼 있어도 fp32로 올린다 (CPU 판정, 노트북 점수와 일치). GPU에서는 autocast가 반정밀도로 돌린다.\n        self.model = AutoModelForSequenceClassification.from_pretrained(model_dir, config=cfg, **kw).float().to(self.device).eval()\n        if mode == \'retrieval\' and isinstance(embedder, str):\n            from sentence_transformers import SentenceTransformer\n            embedder = SentenceTransformer(embedder, device=self.device)\n        self.embedder = embedder\n        self.k, self.max_len, self.chunk_words, self.threshold = k, max_len, chunk_words, threshold\n        self.thresholds, self.threshold_key = thresholds or {}, threshold_key\n        self.default_stride = default_stride\n        self._lock = threading.Lock()     # 모델 1개를 여러 요청이 공유하므로 직렬화\n\n    def threshold_for(self, stride, key=None):\n        """재판정 간격별 임계값 (설정 파일 thresholds[간격][key]). 없으면 기본 임계값."""\n        return (self.thresholds.get(_stride_key(stride)) or {}).get(key or self.threshold_key, self.threshold)\n\n    def _embed(self, texts):\n        return self.embedder.encode(texts, convert_to_tensor=True, normalize_embeddings=True,\n                                    show_progress_bar=False)\n\n    def prepare(self, reference):\n        t0 = time.perf_counter()\n        if self.mode == \'full\':\n            return RefIndex([], None, (time.perf_counter() - t0) * 1000, reference)\n        chunks = chunk_text(reference, self.chunk_words)\n        with self._lock:\n            emb = self._embed(chunks)\n        return RefIndex(chunks, emb, (time.perf_counter() - t0) * 1000, reference)\n\n    def _pair(self, ref, text, prior):\n        if self.mode == \'full\':\n            return ref.text, (prior or \'\') + text\n        e = self._embed([text])[0]\n        top = torch.topk(ref.emb @ e, min(self.k, ref.emb.shape[0])).indices.tolist()\n        return \' ... \'.join(ref.chunks[i] for i in sorted(top)), text\n\n    @torch.no_grad()\n    def judge(self, ref, text, prior=\'\', threshold=None):\n        """text: 판정할 문장(또는 문장 앞부분, 앞뒤 공백 제거), prior: 응답에서 그 문장 앞까지의 원문."""\n        t0 = time.perf_counter()\n        with self._lock:\n            a, b = self._pair(ref, text, prior)\n            enc = self.tok([a], [b], truncation=\'only_first\', max_length=self.max_len,\n                           return_tensors=\'pt\').to(self.device)\n            with torch.autocast(self.device, enabled=(self.device == \'cuda\')):\n                score = float(self.model(**enc).logits.squeeze().float().cpu())\n            if self.device == \'cuda\':\n                torch.cuda.synchronize()\n        thr = self.threshold if threshold is None else threshold\n        return Verdict(score, score >= thr, (time.perf_counter() - t0) * 1000)\n\n\ndef load_judge(settings):\n    """판정기 폴더의 judge_config.json을 읽어 StudentJudge를 만든다. 환경 변수로 지정한 값이 설정 파일보다 우선한다."""\n    cfg = read_config(settings.judge_model_dir)\n    pick = lambda env, key, default: env if env is not None else cfg.get(key, default)\n    dflt = (cfg.get(\'default\') or {})\n    kind = dflt.get(\'kind\', \'end\')\n    return StudentJudge(settings.judge_model_dir,\n                        embedder=pick(settings.judge_embedder, \'embedder\', \'sentence-transformers/all-MiniLM-L6-v2\'),\n                        k=pick(settings.judge_k, \'k\', 8), max_len=pick(settings.judge_max_len, \'max_len\', 512),\n                        chunk_words=pick(settings.judge_chunk_words, \'chunk_words\', 40),\n                        threshold=pick(settings.judge_threshold, \'threshold\', 0.0), device=settings.judge_device,\n                        mode=input_mode(cfg),\n                        # 임계값을 환경 변수로 고정하면 간격별 임계값은 쓰지 않는다\n                        thresholds={} if settings.judge_threshold is not None else cfg.get(\'thresholds\', {}),\n                        threshold_key=dflt.get(\'threshold\', \'svc10_orig\'),\n                        default_stride=0 if kind == \'end\' else int(kind))\n\n\nclass LexicalJudge:\n    """모델 없이 파이프라인을 시험하기 위한 대체 판정기 (테스트·데모 전용).\n    근거에 없는 단어 비율을 점수로 쓴다. 판별력은 없다고 봐야 한다."""\n\n    default_stride = 0\n\n    def __init__(self, threshold=0.5, delay_ms=0.0, fail=None):\n        self.threshold, self.delay_ms = threshold, delay_ms\n        self.fail = fail          # 테스트용: 이 함수가 True를 돌려주는 판정에서 예외를 낸다\n\n    def threshold_for(self, stride, key=None):\n        return self.threshold\n\n    def prepare(self, reference):\n        return RefIndex([reference], {w.lower().strip(\'.,\') for w in reference.split()}, 0.0, reference)\n\n    def judge(self, ref, text, prior=\'\', threshold=None):\n        t0 = time.perf_counter()\n        if self.fail is not None and self.fail(text):\n            raise RuntimeError(\'judge failure (test)\')\n        if self.delay_ms:\n            time.sleep(self.delay_ms / 1000)\n        words = [w.lower().strip(\'.,\') for w in text.split() if w.strip(\'.,\')]\n        score = sum(w not in ref.emb for w in words) / max(len(words), 1)\n        thr = self.threshold if threshold is None else threshold\n        return Verdict(score, score >= thr, (time.perf_counter() - t0) * 1000)\n', 'store': '"""컬렉션 저장소 (SPEC §2.5, §2.6).\n\n관리자·개발자가 등록한 문서를 조각으로 나눠 임베딩과 함께 저장하고, 질문으로 검색한다.\n게이트웨이는 찾은 조각을 LLM 프롬프트에 넣고, **같은 조각을 판정 근거로** 쓴다(LLM이 본 근거 = 판정 근거).\n\n  Store 인터페이스\n    add(collection, text, title=\'\', doc_id=None) -> dict(doc_id, title, n_chunks, n_chars)\n    list(collection) -> [dict(doc_id, title, n_chunks, n_chars, created)]\n    delete(collection, doc_id) -> bool\n    collections() -> [dict(collection, n_docs)]\n    search(collection, query, k) -> [Evidence]       # 점수 내림차순\n\n처음 구현은 SQLite 파일 하나 + 임베딩 배열(코사인 유사도 전수 검색)이다. 규모가 커지면 같은 인터페이스로 벡터 DB로 바꾼다.\n임베더는 encode(texts) -> 정규화된 벡터 배열을 돌려주는 객체(sentence-transformers 호환)면 된다.\n"""\nimport sqlite3\nimport threading\nimport time\nimport uuid\nfrom dataclasses import asdict, dataclass\n\nimport numpy as np\n\nSCHEMA = \'\'\'\nCREATE TABLE IF NOT EXISTS documents (\n    collection TEXT NOT NULL, doc_id TEXT NOT NULL, title TEXT, text TEXT NOT NULL, created REAL NOT NULL,\n    PRIMARY KEY (collection, doc_id));\nCREATE TABLE IF NOT EXISTS chunks (\n    collection TEXT NOT NULL, doc_id TEXT NOT NULL, idx INTEGER NOT NULL, text TEXT NOT NULL, emb BLOB NOT NULL,\n    PRIMARY KEY (collection, doc_id, idx));\n\'\'\'\n\n\n@dataclass\nclass Evidence:\n    doc_id: str\n    title: str\n    chunk_idx: int\n    text: str\n    score: float\n\n    def to_dict(self):\n        return asdict(self)\n\n\ndef split_words(text, n, overlap=0):\n    """n단어 조각(겹침 overlap단어). 빈 문서는 조각 없음."""\n    w = text.split()\n    if not w:\n        return []\n    step = max(1, n - overlap)\n    return [\' \'.join(w[i:i + n]) for i in range(0, max(len(w) - overlap, 1), step)]\n\n\nclass SQLiteStore:\n    def __init__(self, path, embedder, chunk_words=120, overlap=20):\n        self.path, self.embedder = path, embedder\n        self.chunk_words, self.overlap = chunk_words, overlap\n        self._db = sqlite3.connect(path, check_same_thread=False)\n        self._db.executescript(SCHEMA)\n        self._lock = threading.Lock()\n        self._cache = {}          # collection -> (meta 리스트, 임베딩 행렬)\n\n    def _embed(self, texts):\n        v = self.embedder.encode(texts, normalize_embeddings=True, show_progress_bar=False, convert_to_numpy=True)\n        return np.asarray(v, dtype=np.float32).reshape(len(texts), -1)\n\n    def add(self, collection, text, title=\'\', doc_id=None):\n        if not text or not text.strip():\n            raise ValueError(\'document text is empty\')\n        doc_id = doc_id or uuid.uuid4().hex[:16]\n        chunks = split_words(text, self.chunk_words, self.overlap)\n        emb = self._embed(chunks)\n        with self._lock, self._db:\n            self._db.execute(\'DELETE FROM chunks WHERE collection=? AND doc_id=?\', (collection, doc_id))   # 같은 id면 교체\n            self._db.execute(\'INSERT OR REPLACE INTO documents VALUES (?,?,?,?,?)\',\n                             (collection, doc_id, title or \'\', text, time.time()))\n            self._db.executemany(\'INSERT INTO chunks VALUES (?,?,?,?,?)\',\n                                 [(collection, doc_id, i, c, e.tobytes()) for i, (c, e) in enumerate(zip(chunks, emb))])\n            self._cache.pop(collection, None)\n        return {\'doc_id\': doc_id, \'title\': title or \'\', \'n_chunks\': len(chunks), \'n_chars\': len(text)}\n\n    def list(self, collection):\n        with self._lock:\n            rows = self._db.execute(\n                \'SELECT d.doc_id, d.title, length(d.text), d.created, \'\n                \'(SELECT count(*) FROM chunks c WHERE c.collection=d.collection AND c.doc_id=d.doc_id) \'\n                \'FROM documents d WHERE d.collection=? ORDER BY d.created\', (collection,)).fetchall()\n        return [{\'doc_id\': r[0], \'title\': r[1], \'n_chars\': r[2], \'created\': r[3], \'n_chunks\': r[4]} for r in rows]\n\n    def delete(self, collection, doc_id):\n        with self._lock, self._db:\n            n = self._db.execute(\'DELETE FROM documents WHERE collection=? AND doc_id=?\', (collection, doc_id)).rowcount\n            self._db.execute(\'DELETE FROM chunks WHERE collection=? AND doc_id=?\', (collection, doc_id))\n            self._cache.pop(collection, None)\n        return n > 0\n\n    def collections(self):\n        with self._lock:\n            rows = self._db.execute(\'SELECT collection, count(*) FROM documents GROUP BY collection ORDER BY collection\').fetchall()\n        return [{\'collection\': c, \'n_docs\': n} for c, n in rows]\n\n    def exists(self, collection):\n        with self._lock:\n            return self._db.execute(\'SELECT 1 FROM documents WHERE collection=? LIMIT 1\', (collection,)).fetchone() is not None\n\n    def _matrix(self, collection):\n        with self._lock:\n            if collection not in self._cache:\n                rows = self._db.execute(\n                    \'SELECT c.doc_id, d.title, c.idx, c.text, c.emb FROM chunks c JOIN documents d \'\n                    \'ON c.collection=d.collection AND c.doc_id=d.doc_id WHERE c.collection=? ORDER BY d.created, c.idx\',\n                    (collection,)).fetchall()\n                meta = [r[:4] for r in rows]\n                mat = np.stack([np.frombuffer(r[4], dtype=np.float32) for r in rows]) if rows else np.zeros((0, 1), np.float32)\n                self._cache[collection] = (meta, mat)\n            return self._cache[collection]\n\n    def search(self, collection, query, k):\n        meta, mat = self._matrix(collection)\n        if not meta or not query.strip():\n            return []\n        sims = mat @ self._embed([query])[0]\n        top = np.argsort(-sims)[:k]\n        return [Evidence(meta[i][0], meta[i][1], meta[i][2], meta[i][3], float(sims[i])) for i in top]\n\n    def close(self):\n        self._db.close()\n', 'context': '"""요청에서 근거 문서(RAG 컨텍스트)와 검증 옵션을 꺼낸다.\n\n우선순위\n1. 최상위 필드 `verification.context` (문자열 또는 문자열 리스트)\n   - OpenAI SDK에서는 `extra_body={"verification": {...}}` 로 보낸다.\n2. 메시지 본문 안의 `<context>...</context>` 태그 (RAG 서비스가 흔히 쓰는 형식)\n3. `verification.collection` — 게이트웨이에 등록된 컬렉션에서 검색 (app이 처리, SPEC §2.6)\n4. 모두 없으면 검증하지 않는다 (mode=off).\n\n`verification` 필드는 실제 LLM API가 모르는 필드이므로 전달 전에 제거한다.\n"""\nimport copy\nimport re\n\nCTX_TAG = re.compile(r\'<context>(.*?)</context>\', re.S | re.I)\nMODES = {\'pass\', \'hold\', \'off\'}\nACTIONS = {\'flag\', \'cut\'}\n\n\ndef _content_text(content):\n    if isinstance(content, str):\n        return content\n    if isinstance(content, list):          # [{"type":"text","text":...}, ...]\n        return \'\\n\'.join(p.get(\'text\', \'\') for p in content if isinstance(p, dict))\n    return \'\'\n\n\nRAG_INSTRUCTION = (\'Answer using only the reference documents below. \'\n                   \'If they do not contain the answer, say that you do not know.\')\n\n\ndef last_user_text(body):\n    users = [m for m in body.get(\'messages\', []) if m.get(\'role\') == \'user\']\n    return _content_text(users[-1].get(\'content\')) if users else \'\'\n\n\ndef rag_messages(messages, evidence):\n    """근거 조각을 담은 시스템 메시지를 맨 앞에 더한다. 서비스가 보낸 메시지는 바꾸지 않는다 (SPEC §2.6)."""\n    block = \'\\n\\n\'.join(f\'[{i + 1}] {e.text}\' for i, e in enumerate(evidence))\n    return [{\'role\': \'system\', \'content\': f\'{RAG_INSTRUCTION}\\n\\n<documents>\\n{block}\\n</documents>\'}] + list(messages)\n\n\ndef rag_reference(evidence):\n    """판정 근거 = LLM에 넣은 조각과 같은 텍스트."""\n    return \'\\n\\n\'.join(e.text for e in evidence)\n\n\ndef extract(body, default_mode=\'pass\', default_action=\'flag\', default_stride=0, with_stride=False, with_opts=False):\n    """반환: (upstream으로 보낼 body, reference 문자열 또는 None, mode, action[, stride][, opts])\n    with_stride=True면 앞부분 재판정 간격(stride)도 함께 돌려준다.\n    with_opts=True면 원래 verification 옵션도 돌려준다(collection 등). collection이 있고 근거가 없으면 mode를 off로 바꾸지 않는다."""\n    body = copy.deepcopy(body)\n    opts = body.pop(\'verification\', None) or {}\n\n    ref = opts.get(\'context\')\n    if isinstance(ref, list):\n        ref = \'\\n\\n\'.join(str(x) for x in ref)\n    if not ref:\n        found = []\n        for m in body.get(\'messages\', []):\n            found += CTX_TAG.findall(_content_text(m.get(\'content\')))\n        ref = \'\\n\\n\'.join(s.strip() for s in found) or None\n\n    mode = opts.get(\'mode\', default_mode)\n    action = opts.get(\'action\', default_action)\n    if mode not in MODES:\n        raise ValueError(f\'verification.mode must be one of {sorted(MODES)}\')\n    if action not in ACTIONS:\n        raise ValueError(f\'verification.action must be one of {sorted(ACTIONS)}\')\n    stride = opts.get(\'stride\', default_stride)\n    if not isinstance(stride, int) or isinstance(stride, bool) or stride < 0:\n        raise ValueError(\'verification.stride must be a non-negative integer\')\n    coll = opts.get(\'collection\')\n    if coll is not None and (not isinstance(coll, str) or not coll):\n        raise ValueError(\'verification.collection must be a non-empty string\')\n    if not ref and not coll:\n        mode = \'off\'\n    out = (body, ref, mode, action) + ((stride,) if with_stride else ()) + ((opts,) if with_opts else ())\n    return out\n'}
import os, sys, json, time, random, subprocess, shutil, pickle
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'sentence-transformers', 'tiktoken', 'pysbd', '-U', 'transformers>=4.48'], check=True)
subprocess.run('git clone --depth 1 https://github.com/ParticleMedia/RAGTruth.git 2>/dev/null || true', shell=True)
subprocess.run('git clone --depth 1 https://github.com/vectara/FaithBench.git 2>/dev/null || true', shell=True)
import numpy as np, torch
from google.colab import drive
drive.mount('/content/drive')
# 게이트웨이 코드 그대로 (패키지 __init__은 FastAPI 앱을 불러오므로 비워 둔다)
os.makedirs('/content/hg/hallugate', exist_ok=True)
open('/content/hg/hallugate/__init__.py', 'w').write('')
for n, s in SRC.items(): open(f'/content/hg/hallugate/{n}.py', 'w').write(s)
sys.path.insert(0, '/content/hg')
from hallugate.judge import StudentJudge, read_config, input_mode
from hallugate.store import SQLiteStore
from hallugate.context import rag_messages, rag_reference

OUT = '/content/drive/MyDrive/Colab Notebooks/HalluGate'
F_DIR, I_DIR = os.path.join(OUT, 'final', 'judge_base'), os.path.join(OUT, 'i2', 'judge_i2')
X2_DIR, K2_DIR = os.path.join(OUT, 'x2'), os.path.join(OUT, 'k2')
os.makedirs(X2_DIR, exist_ok=True)
for d in [F_DIR, I_DIR]:
    assert os.path.exists(os.path.join(d, 'judge_config.json')), f'{d}/judge_config.json 없음'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
GPU = torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'cpu'
COMBO = {'F_mean': -3.6794357299804688, 'F_std': 2.1870224475860596, 'I_mean': -4.456174373626709, 'I_std': 2.400838851928711, 'threshold': 2.991638706756592}
def save_json(o, p): json.dump(o, open(p + '.tmp', 'w'), ensure_ascii=False); os.replace(p + '.tmp', p)
def load_json(p): return json.load(open(p)) if os.path.exists(p) else None
print('device', GPU, '| 조합 상수(I3)', COMBO)

## 2. FaithBench와 RAGTruth 학습 근거 겹침

In [ ]:
# 판정 입력: RAGTruth train 응답의 문장 끝 + 8토큰 지점 (게이트웨이와 같은 cl100k 토큰, 문장 시작부터)
import pysbd, tiktoken
ENC = tiktoken.get_encoding('cl100k_base')
SEG = pysbd.Segmenter(language='en', clean=False, char_span=True)

def build_reference(src):
    si, tt = src['source_info'], src['task_type']
    if tt == 'QA':
        return f"Question: {si.get('question','')}\n{si.get('passages','')}"
    if tt == 'Summary':
        return si if isinstance(si, str) else json.dumps(si, ensure_ascii=False)
    return '\n'.join(f"{k}: {json.dumps(v, ensure_ascii=False) if isinstance(v,(dict,list)) else v}"
                     for k, v in (si.items() if isinstance(si, dict) else []))

def points_of(text, s=8):
    # (문장 시작 글자, 지점 글자 끝) 목록: 문장마다 s토큰째 + 문장 끝
    ids = ENC.encode(text, disallowed_special=())
    _, starts = ENC.decode_with_offsets(ids)
    ends = starts[1:] + [len(text)]
    out = []
    for sp in SEG.segment(text):
        if not sp.sent.strip(): continue
        a = sp.start + (len(sp.sent) - len(sp.sent.lstrip()))
        tok = [e for st, e in zip(starts, ends) if a <= st < sp.end or (st < a < e)]
        for k, e in enumerate(tok, 1):
            if k % s == 0 or k == len(tok):
                out.append((a, min(e, sp.end)))
    return out

def make_inputs(responses, sources, n_resp=300, n_pts=300, seed=2027):
    rs = random.Random(seed)
    pool = [r for r in responses if r['split'] == 'train']
    cands = []
    for r in rs.sample(pool, min(n_resp, len(pool))):
        ref = build_reference(sources[r['source_id']])
        for a, cut in points_of(r['response']):
            text = r['response'][a:cut].strip()
            if text: cands.append({'ref': ref, 'text': text, 'prior': r['response'][:a], 'task': sources[r['source_id']]['task_type']})
    return rs.sample(cands, min(n_pts, len(cands)))

import glob, re
FB = []
for f in sorted(glob.glob('FaithBench/data_for_release/batch_*.json'), key=lambda p: int(re.findall(r'(\d+)', p)[-1])):
    FB += json.load(open(f))['samples']
SEV = {'Benign': 1, 'Questionable': 2}
def sev(l): return 3 if l.startswith('Unwanted') else SEV.get(l, 0)

def fb_label(s):
    # 최악 등급(FaithBench binarize.py의 worst): 3 = Unwanted가 하나라도, 2 = Questionable, 1 = Benign만, 0 = 표시 없음
    ls = {l for a in s['annotations'] for l in a['label']}
    return max([sev(l) for l in ls], default=0)

def fb_onset(s):
    # 첫 Unwanted 구간의 요약 글자 위치 (없으면 None)
    st = [a['summary_start'] for a in s['annotations'] if any(l.startswith('Unwanted') for l in a['label']) and a.get('summary_start') is not None]
    return min(st) if st else None

def ngrams(t, n=8):
    w = re.findall(r'\w+', t.lower())
    return {tuple(w[i:i + n]) for i in range(max(len(w) - n + 1, 0))}

def overlap_flags(fb_sources, rt_sources, thr=0.5):
    # FaithBench 근거가 RAGTruth train 요약 근거와 겹치는지: 8-gram 포함률 ≥ 0.5
    rt = [ngrams(t) for t in rt_sources]
    out = {}
    for s in fb_sources:
        g = ngrams(s)
        out[s] = max([len(g & r) / max(len(g), 1) for r in rt], default=0.0) >= thr if g else False
    return out

assert len(FB) == 750, len(FB)
responses = [json.loads(l) for l in open('RAGTruth/dataset/response.jsonl')]
sources = {json.loads(l)['source_id']: json.loads(l) for l in open('RAGTruth/dataset/source_info.jsonl')}
_train_sids = {r['source_id'] for r in responses if r['split'] == 'train'}          # test 분할 근거는 보지 않는다
RT_SUM = [build_reference(sources[s]) for s in _train_sids if sources[s]['task_type'] == 'Summary']
OVERLAP = overlap_flags(sorted({s['source'] for s in FB}), RT_SUM)
print(f"FaithBench 요약 {len(FB)}개 | 근거 {len(OVERLAP)}개 중 RAGTruth 학습 요약 근거와 겹침 {sum(OVERLAP.values())}개")
import collections
print('최악 등급(0 표시 없음, 1 Benign, 2 Questionable, 3 Unwanted):', dict(sorted(collections.Counter(fb_label(s) for s in FB).items())))

## 3. 판정 — 기존 조합(필수), K2 새 조합(있으면)

In [ ]:
import contextlib, threading
from concurrent.futures import ThreadPoolExecutor

def make_judge(d, device):
    # load_judge와 같은 값으로 (판정기 폴더의 judge_config.json)
    cfg = read_config(d)
    return StudentJudge(d, embedder=cfg.get('embedder', 'sentence-transformers/all-MiniLM-L6-v2'), k=cfg.get('k', 8),
                        max_len=cfg.get('max_len', 512), chunk_words=cfg.get('chunk_words', 40), device=device,
                        mode=input_mode(cfg))

def combine(sf, si):
    # I3 조합: 두 점수를 cal-L 멀쩡한 응답 기준으로 표준화해 평균
    return ((sf - COMBO['F_mean']) / COMBO['F_std'] + (si - COMBO['I_mean']) / COMBO['I_std']) / 2

def judge_seq(jF, jI, refF, refI, x):
    # 차례로: 게이트웨이 judge() 두 번 (각각 토크나이즈·검색·순전파·동기화 포함)
    return jF.judge(refF, x['text'], x['prior']).score, jI.judge(refI, x['text'], x['prior']).score

STREAMS = {}
def _score_on_stream(j, ref, text, prior):
    # judge()와 같은 계산을 판정기 전용 CUDA 스트림에서. 동기화는 그 스트림만 (.cpu()가 현재 스트림을 기다림)
    ctx = torch.cuda.stream(STREAMS.setdefault(id(j), torch.cuda.Stream())) if j.device == 'cuda' else contextlib.nullcontext()
    with ctx, torch.no_grad():
        a, b = j._pair(ref, text, prior)
        enc = j.tok([a], [b], truncation='only_first', max_length=j.max_len, return_tensors='pt').to(j.device)
        with torch.autocast(j.device, enabled=(j.device == 'cuda')):
            return float(j.model(**enc).logits.squeeze().float().cpu())

POOL = ThreadPoolExecutor(2)
def judge_par(jF, jI, refF, refI, x):
    fF = POOL.submit(_score_on_stream, jF, refF, x['text'], x['prior'])
    fI = POOL.submit(_score_on_stream, jI, refI, x['text'], x['prior'])
    return fF.result(), fI.result()

def bench(fn, jF, jI, inputs, refs, warm=10):
    for x in inputs[:warm]: fn(jF, jI, *refs[x['ref']], x)
    ms, sc = [], []
    for x in inputs:
        t0 = time.perf_counter()
        sf, si = fn(jF, jI, *refs[x['ref']], x)
        c = combine(sf, si)
        ms.append((time.perf_counter() - t0) * 1000); sc.append((sf, si, c))
    return np.array(ms), np.array(sc)

def bench_one(j, inputs, refs, which, warm=10):
    for x in inputs[:warm]: j.judge(refs[x['ref']][which], x['text'], x['prior'])
    return np.array([j.judge(refs[x['ref']][which], x['text'], x['prior']).judge_ms for x in inputs])

def score_response(jF, jI, refs, text, fn):
    # 판정 지점(문장마다 8토큰째 + 문장 끝)마다 (문장 시작, 지점 끝, F1 base, I2, 조합)
    out = []
    for a, cut in points_of(text):
        t = text[a:cut].strip()
        if not t: continue
        sf, si = fn(jF, jI, refs[0], refs[1], {'text': t, 'prior': text[:a]})
        out.append((a, cut, sf, si, combine(sf, si)))
    return out

def gen_status(g):
    if g is None: return 'nogen'
    if g['stop'] == 'refusal': return 'refusal'
    if not g['text'].strip() or not sentences(g['text']): return 'empty'
    return 'valid'

jF, jI = make_judge(F_DIR, DEVICE), make_judge(I_DIR, DEVICE)
FN = judge_par if DEVICE == 'cuda' else judge_seq
THR = {'c': COMBO['threshold']}
K2_JUDGE, K2_SUM = os.path.join(K2_DIR, 'judge_base'), os.path.join(K2_DIR, 'k2_summary.json')
HAS_K2 = os.path.exists(os.path.join(K2_JUDGE, 'judge_config.json')) and os.path.exists(K2_SUM)
if HAS_K2:
    K2S = load_json(K2_SUM); CN = K2S['combo_new']
    THR['n'] = K2S['thr']['new']['A']; THR['nB'] = K2S['thr']['new']['B']; THR['cB'] = K2S['thr']['old']['B']
    jK = make_judge(K2_JUDGE, DEVICE)
print('K2 판정기', '있음' if HAS_K2 else '없음(기존 조합만)', '| 임계값', THR)
sp = os.path.join(X2_DIR, 'scores_x2.json')
SC = load_json(sp) or {}
t0 = time.time()
for i, s in enumerate(FB):
    if str(i) in SC and (not HAS_K2 or all(len(p) == 6 for p in SC[str(i)])): continue
    src, summ = s['source'], s['summary']
    pts = [list(p) for p in score_response(jF, jI, (jF.prepare(src), jI.prepare(src)), summ, FN)]
    if HAS_K2:
        rk = jK.prepare(src)
        for p in pts:
            sk = jK.judge(rk, summ[p[0]:p[1]].strip(), summ[:p[0]]).score
            p.append(((sk - CN['F_mean']) / CN['F_std'] + (p[3] - CN['I_mean']) / CN['I_std']) / 2)
    SC[str(i)] = pts
    if (i + 1) % 50 == 0:
        save_json(SC, sp); print(f'  판정 {i + 1}/{len(FB)} ({time.time() - t0:.0f}s)', flush=True)
save_json(SC, sp)

## 4. 지표와 해석 규칙

In [ ]:
from sklearn.metrics import roc_auc_score

def auroc(y, s):
    y, s = np.asarray(y, bool), np.asarray(s, float)
    if y.all() or (~y).all(): return float('nan')
    return float(roc_auc_score(y, s))

def x2_rows(FB, SC, thr, overlap):
    # 요약 하나 = 행. hal = Unwanted 있음, clean = 표시 없음 또는 Benign만, Questionable이 최악이면 주 분석에서 뺌
    rows = []
    sid = {t: k for k, t in enumerate(sorted({x['source'] for x in FB}))}
    for i, s in enumerate(FB):
        pts = SC.get(str(i), [])
        lab = fb_label(s); on = fb_onset(s)
        r = {'i': i, 'src': sid[s['source']], 'summarizer': s['metadata']['summarizer'], 'worst': lab,
             'hal': lab == 3, 'primary': lab != 2 and not overlap.get(s['source'], False), 'overlap': overlap.get(s['source'], False),
             'n_pts': len(pts), 'onset': on,
             'hhem21': 1 - s['metadata']['hhem-2.1'], 'hhem21_en': 1 - s['metadata']['hhem-2.1-english']}
        for b in ['gpt-3.5-turbo', 'gpt-4-turbo', 'gpt_4o', 'trueteacher', 'true_nli']:
            v = s['metadata'].get(b); r[f'pred_{b}'] = None if v is None else int(v == 0)      # 1 = 환각이라고 예측
        for name, col in [('c', 4), ('f', 2), ('i', 3), ('n', 5)]:
            vals = [p[col] for p in pts if len(p) > col and p[col] is not None]
            r[f'max_{name}'] = max(vals) if vals else np.nan
        for name, col in [('c', 4), ('n', 5)]:
            t = thr.get(name)
            if t is None or np.isnan(r[f'max_{name}']): continue
            first = next((p[0] for p in pts if p[col] is not None and p[col] >= t), None)
            r[f'flag_{name}'] = first is not None
            h0 = max([p[0] for p in pts if p[0] <= on], default=0) if on is not None else None     # 첫 Unwanted 구간이 든 문장의 시작
            r[f'strict_{name}'] = bool(on is not None and first is not None and first >= h0)
        rows.append(r)
    return pd.DataFrame(rows)

def balanced_acc(y, pred):
    y, p = np.asarray(y, bool), np.asarray(pred, bool)
    return float((p[y].mean() + (~p[~y]).mean()) / 2)

def cluster_boot(df, fn, n=2000, seed=2034):
    # 같은 근거의 요약 10개가 묶여 있으므로 근거 단위로 다시 뽑는다
    srcs = df.src.unique(); rng = np.random.default_rng(seed)
    g = {s: df[df.src == s] for s in srcs}
    vals = []
    for _ in range(n):
        d = pd.concat([g[s] for s in rng.choice(srcs, len(srcs))])
        v = fn(d)
        if v == v: vals.append(v)
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

def x2_summary(df):
    P = df[df.primary]
    out = {'n_all': len(df), 'n_primary': len(P), 'n_hal': int(P.hal.sum()), 'n_clean': int((~P.hal).sum()),
           'n_overlap': int(df.overlap.sum()), 'n_questionable_worst': int((df.worst == 2).sum())}
    cols = [c for c in ['max_c', 'max_f', 'max_i', 'max_n', 'hhem21', 'hhem21_en'] if c in P and P[c].notna().all()]
    out['auroc'] = {c: auroc(P.hal, P[c]) for c in cols}
    out['auroc_ci'] = {c: cluster_boot(P, lambda d, c=c: auroc(d.hal, d[c])) for c in cols}
    out['auroc_diff_c_vs_hhem21'] = {'delta': out['auroc']['max_c'] - out['auroc']['hhem21'],
                                     'ci': cluster_boot(P, lambda d: auroc(d.hal, d.max_c) - auroc(d.hal, d.hhem21))}
    if 'max_n' in cols:
        out['auroc_diff_n_vs_c'] = {'delta': out['auroc']['max_n'] - out['auroc']['max_c'],
                                    'ci': cluster_boot(P, lambda d: auroc(d.hal, d.max_n) - auroc(d.hal, d.max_c))}
    out['bacc'] = {}
    for c in ['flag_c', 'flag_n'] + [k for k in P.columns if k.startswith('pred_')]:
        if c in P and P[c].notna().all(): out['bacc'][c] = balanced_acc(P.hal, P[c].astype(bool))
    for c in ['flag_c', 'flag_n']:
        if c in P and P[c].notna().all():
            out[f'{c}_hal_rate'] = float(P[P.hal][c].astype(bool).mean()); out[f'{c}_clean_rate'] = float(P[~P.hal][c].astype(bool).mean())
    for c in ['strict_c', 'strict_n']:
        if c in P and P[c].notna().all(): out[f'{c}_rate'] = float(P[P.hal][c].astype(bool).mean())
    out['by_summarizer_auroc_c'] = {m: auroc(d.hal, d.max_c) for m, d in P.groupby('summarizer')}
    out['by_summarizer_n_hal'] = {m: int(d.hal.sum()) for m, d in P.groupby('summarizer')}
    Q = df[~df.overlap]                                  # 보고: Questionable도 환각으로
    out['auroc_q_as_hal'] = {c: auroc(Q.worst >= 2, Q[c]) for c in cols}
    return out

DF = x2_rows(FB, SC, THR, OVERLAP)
if HAS_K2:                                     # 임계값 B 표시도 보고
    for name, col, key in [('cB', 4, 'cB'), ('nB', 5, 'nB')]:
        DF[f'flag_{name}'] = [any(p[col] >= THR[key] for p in SC[str(i)]) for i in DF.i]
X = x2_summary(DF)
if HAS_K2:
    P = DF[DF.primary]
    for c in ['flag_cB', 'flag_nB']:
        X['bacc'][c] = balanced_acc(P.hal, P[c]); X[f'{c}_hal_rate'] = float(P[P.hal][c].mean()); X[f'{c}_clean_rate'] = float(P[~P.hal][c].mean())
print(f"주 분석: 요약 {X['n_primary']}개 (환각 {X['n_hal']}, 멀쩡 {X['n_clean']}) | 제외: Questionable 최악 {X['n_questionable_worst']}, 겹침 근거 요약 {X['n_overlap']}")
for c, v in X['auroc'].items(): print(f"  AUROC {c:10} {v:.3f} [{X['auroc_ci'][c][0]:.3f}, {X['auroc_ci'][c][1]:.3f}]")
print('  균형 정확도', {k: round(v, 3) for k, v in X['bacc'].items()})
print(f"  임계값 A 기존 조합: 환각 탐지 {X.get('flag_c_hal_rate', float('nan')):.3f}, 멀쩡한 응답 표시 {X.get('flag_c_clean_rate', float('nan')):.3f}, 첫 Unwanted 이후 탐지 {X.get('strict_c_rate', float('nan')):.3f}")
print('  생성 모델별 기존 조합 AUROC', {m.split('/')[-1]: round(v, 3) for m, v in X['by_summarizer_auroc_c'].items()})
d = X['auroc_diff_c_vs_hhem21']; lo, hi = d['ci']
verdict = '앞섬' if lo > 0 else ('뒤짐' if hi < 0 else '비슷')
disc = X['auroc_ci']['max_c'][0] > 0.5
print(f"\n=== X2 해석: 기존 조합 − HHEM-2.1 AUROC {d['delta']:+.3f} [{lo:+.3f}, {hi:+.3f}] → 공개 판정기(HHEM-2.1)보다 {verdict} ===")
print(f"=== 기존 조합 AUROC 하한 {X['auroc_ci']['max_c'][0]:.3f} → {'판별력 있음(> 0.5)' if disc else 'FaithBench에서 판별력 확인 안 됨'} ===")
if HAS_K2:
    e = X['auroc_diff_n_vs_c']; print(f"(보고) K2 새 조합 − 기존 조합 AUROC {e['delta']:+.3f} [{e['ci'][0]:+.3f}, {e['ci'][1]:+.3f}]")
X['verdict_vs_hhem21'], X['discriminates'], X['has_k2'], X['thr'] = verdict, bool(disc), HAS_K2, THR

## 5. 저장

In [ ]:
LOCAL = '/content/x2_local'; os.makedirs(LOCAL, exist_ok=True)
DF.to_csv(os.path.join(LOCAL, 'x2_rows.csv'), index=False)        # 원문 없음
json.dump(dict(X, gpu=GPU), open(os.path.join(LOCAL, 'x2_summary.json'), 'w'), ensure_ascii=False, indent=1, default=float)
for f in os.listdir(LOCAL):
    try: shutil.copy(os.path.join(LOCAL, f), os.path.join(X2_DIR, f))
    except OSError as e: print('드라이브 저장 실패', f, e)
zp = shutil.make_archive('/content/x2_results', 'zip', LOCAL)
from google.colab import files
files.download(zp)